In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_guide.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

C:\Users\samet\AppData\Local\Temp\ipykernel_1908\2523830466.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
C:\Users\samet\OneDrive\Masaüstü\Projects\tutorial-agentic-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
print(len(pages))
print(pages[2].page_content[:500])

9
Telecom Technical Reference Guide  - Internal Use Only
2. Troubleshooting Connectivity Issues
Connectivity problems are the most common category of customer complaints. A structured diagnostic approach
resolves the majority of cases without escalation.
Step 1  - Verify signal strength. Open the device's status bar or dial *3001#12345#* (iOS) or use a network signal
app (Android) to view the raw signal level in dBm. A signal above -85 dBm is good; between -85 and -100 dBm is
marginal; below -100 


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", "."," "],
)

chunks = splitter.split_documents(pages)

len(chunks)

37

In [8]:
chunks[2].page_content

'downloads at speeds of 1-14 Mbps. This generation established the foundation for smartphone adoption\nworldwide.\n4G (LTE) networks delivered a major leap forward, with typical download speeds of 20-150 Mbps and latency\nbelow 50 ms. LTE networks are fully IP-based, meaning voice calls are transmitted as data (VoLTE) rather than\nover a separate circuit-switched channel. LTE-Advanced (LTE-A) uses carrier aggregation to combine multiple\nfrequency bands, pushing speeds beyond 300 Mbps.\n5G networks represent the current frontier. Deploying across sub-6 GHz and millimetre-wave (mmWave)'

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = Chroma.from_documents(chunks, embeddings)

print(f"Vector store ready. {vector_store._collection.count()} vectors stored.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1974.97it/s]


Vector store ready. 37 vectors stored.


RETRIVER 

In [12]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})


test_query = "What is VoLTE and how does it improve call quality?"
retrieved = retriever.invoke(test_query)


for i, doc in enumerate(retrieved, 1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:300])
    print()

--- Chunk 1 ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

--- Chunk 2 ---
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt



RAG CHAIN

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq


# --- Helper: join retrieved chunks into a single context string ---
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)



SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])


llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    reasoning_format="parsed",
)


# --- Assemble the chain ---
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [15]:
question = "How does international roaming work and what charges should I expect?"

print(f"Q: {question}\n")
print("A:", chain.invoke(question))

Q: How does international roaming work and what charges should I expect?

A: International roaming allows your device to connect to partner networks in foreign countries when outside your home network's coverage. Here’s how it works and the charges you may incur:

### **How It Works**  
1. **Authentication**: The visited country’s network verifies your device using signaling protocols (SS7 or Diameter).  
2. **Authorization**: Your home network checks your subscription and approves service.  
3. **Billing Tunneling**: All data, voice, and SMS traffic is routed back to your home network for billing, which may introduce latency.  

### **Charges by Roaming Zones**  
- **Zone A** (EU, UK, Australia, New Zealand): Lowest roaming rates.  
- **Zone B** (USA, Canada, Japan, Singapore): Moderate rates.  
- **Zone C** (Rest of the World): Highest per-MB and per-minute charges.  

**Recommendation**: Always purchase a roaming bundle before traveling to Zone B or C to avoid unexpected high charge